# 1.5 — Casting and Range of Variables 🧋

### AP CSA · Unit 1: Using Objects and Methods
**Boba Cafe Series · Lesson 5**

---

> **Setup note:** every code cell runs on the **IJava kernel** (Java 17+). Check that the kernel picker says *Java*. Each cell declares a class and then calls it with `ClassName.main(null);`.

## Two walls, one door

You've hit the same barrier twice now and been told to wait:

- **Lesson 1.2:** `int price = 5.75;` refused to compile — *possible lossy conversion*.
- **Lesson 1.3:** `25 / 2` gave you `12` and silently threw away the half.

Both have the same fix, and it's called **casting**. Today you get the key.

But casting comes with a warning label, so this lesson has a second half. Java's containers have **limits** — an `int` can only hold numbers up to a certain size, and a `double` can only hold so many digits of precision. Push past either limit and Java doesn't crash, doesn't warn you, and doesn't stop. It just hands you a wrong number and keeps going.

The manager puts it this way:

> "Last month the cup counter read negative four hundred thousand. We've never given away a cup, let alone four hundred thousand of them. Find out what happened."

### What you'll be able to do by the end

| # | Objective | CED reference |
|---|---|---|
| 1 | Cast between `int` and `double` and determine the resulting value | 1.5.A |
| 2 | Explain why casting **truncates** instead of rounding, and round correctly | 1.5.A |
| 3 | Apply cast **precedence** correctly inside arithmetic expressions | 1.5.A |
| 4 | State the range of an `int` and describe when an expression goes **out of range** | 1.5.B |
| 5 | Recognize **overflow** and why it produces no error message | 1.5.B |
| 6 | Describe conditions that limit the **accuracy** of expressions | 1.5.C |

---

## Part 1 — The cast operator

From 1.4 you know Java moves values between types in one direction for free:

| Direction | Name | Needs a cast? | Why |
|---|---|---|---|
| `int` → `double` | **Widening** | No, automatic | Nothing is lost. `7` becomes `7.0` |
| `double` → `int` | **Narrowing** | **Yes, required** | The fractional part must be discarded |

A **cast** is you telling the compiler: *"I know this loses information. Do it anyway. That's on me."*

The syntax is the target type in parentheses, in front of the value:

```java
(int) someDouble
(double) someInt
```

Java's refusal to narrow automatically isn't the compiler being difficult — it's a safety interlock. Casting is how you deliberately disable it.

In [ ]:
public class CastBasics {
    public static void main(String[] args) {
        int scoops = 3;

        double asDouble = scoops;              // WIDENING -- automatic, no cast needed
        System.out.println("int 3 widened to double:  " + asDouble);

        double ounces = 16.8;
        int wholeOunces = (int) ounces;        // NARROWING -- the cast is required
        System.out.println("double 16.8 cast to int:  " + wholeOunces);

        System.out.println("The original is untouched: " + ounces);
    }
}

CastBasics.main(null);

That last line matters: **a cast does not change the original variable.** It produces a new value of a different type. `ounces` is still `16.8` afterwards.

---

## Part 2 — Casting truncates. It does not round.

This is the single most tested fact about casting:

> **`(int)` chops off everything after the decimal point.** It does not round, ever.

So `(int) 5.99` is `5`. Not `6`. The `.99` is discarded without a second thought.

At the cafe: you have 16.8 ounces of tea left and cups hold 1 ounce each. You can fill **16** cups. That 0.8 isn't rounded up into a 17th cup — it just isn't a cup.

### What about negatives?

Truncation always moves **toward zero**, which surprises people. `(int) -5.99` is `-5`, not `-6`. It chops the digits off; it doesn't "round down."

In [ ]:
public class Truncation {
    public static void main(String[] args) {
        System.out.println("(int) 5.99  = " + (int) 5.99);
        System.out.println("(int) 5.50  = " + (int) 5.50);
        System.out.println("(int) 5.01  = " + (int) 5.01);
        System.out.println();
        System.out.println("(int) -5.99 = " + (int) -5.99);
        System.out.println("(int) -5.50 = " + (int) -5.50);
        System.out.println("(int) -5.01 = " + (int) -5.01);
    }
}

Truncation.main(null);

Every positive one gave `5`. Every negative one gave `-5`. Truncation deletes digits — it doesn't move toward negative infinity.

### Rounding on purpose

Sometimes you genuinely want to round. The standard trick is to **add 0.5 before truncating**:

```java
int rounded = (int)(value + 0.5);
```

Why it works: `5.4 + 0.5` is `5.9`, which truncates to `5`. But `5.6 + 0.5` is `6.1`, which truncates to `6`. Adding half pushes anything at or above `.5` over the next whole number before the chop happens.

> This trick works for **positive** numbers. For negatives you'd subtract 0.5 instead. Java also has a built-in `Math.round()`, which is Topic **1.11**.

In [ ]:
public class RoundingTrick {
    public static void main(String[] args) {
        double a = 5.99;
        double b = 5.50;
        double c = 5.01;

        System.out.println("value   truncated   rounded");
        System.out.println(a + "    " + (int) a + "           " + (int)(a + 0.5));
        System.out.println(b + "     " + (int) b + "           " + (int)(b + 0.5));
        System.out.println(c + "    " + (int) c + "           " + (int)(c + 0.5));
    }
}

RoundingTrick.main(null);

---

## Part 3 — Where you put the cast changes everything

Here's the trap that decides a lot of exam points.

> **A cast binds tighter than `*`, `/`, `%`, `+`, and `-`.** It applies to the value *immediately* to its right, not to the whole expression.

Compare these two, using `25 / 2`:

```java
(double) totalOunces / cups      // cast the 25 FIRST, then divide -> 12.5
(double) (totalOunces / cups)    // divide FIRST (int division!), then cast -> 12.0
```

The first one converts `25` to `25.0`, so the division is `double / int`, which produces `12.5`.

The second one has parentheses, so `25 / 2` runs first as **integer division**, producing `12`. Casting after the damage is done just gives you `12.0`. The half ounce was destroyed before the cast ever ran.

**You cannot recover precision that was already thrown away. Cast before the division, never after.**

In [ ]:
public class CastPrecedence {
    public static void main(String[] args) {
        int totalOunces = 25;
        int cups = 2;

        System.out.println("no cast:                   " + (totalOunces / cups));
        System.out.println("(double) totalOunces/cups: " + ((double) totalOunces / cups));
        System.out.println("totalOunces/(double) cups: " + (totalOunces / (double) cups));
        System.out.println("(double)(totalOunces/cups): " + ((double) (totalOunces / cups)));
    }
}

CastPrecedence.main(null);

Lines 2 and 3 both give `12.5` — casting **either** operand is enough, because one `double` makes the whole division a `double` operation (the rule from 1.3).

Line 4 gives `12.0`, which is the wrong answer wearing a decimal point as a disguise. That's the version that looks right in a code review and quietly shorts your customers.

This is the promised fix for the receipt bug from Lesson 1.3:

```java
double ouncesPerCup = (double) totalOunces / cups;   // 12.5
```

---

## Part 4 — How big can an `int` get?

An `int` occupies **32 bits** of memory. That's a fixed number of switches, so there is a hard ceiling and a hard floor on what it can represent.

| Constant | Value |
|---|---|
| `Integer.MAX_VALUE` | 2,147,483,647 |
| `Integer.MIN_VALUE` | −2,147,483,648 |

Both constants are on the official AP Java Quick Reference, so you don't have to memorize the digits — but you should know they're roughly **±2.1 billion**.

Notice the floor is one further from zero than the ceiling. That's not a typo; it's a consequence of zero taking up one of the positive slots.

In [ ]:
public class RangeCheck {
    public static void main(String[] args) {
        System.out.println("Integer.MAX_VALUE = " + Integer.MAX_VALUE);
        System.out.println("Integer.MIN_VALUE = " + Integer.MIN_VALUE);

        int cupsPerDay = 400;
        int daysOpen   = 365;
        System.out.println("Cups per year:      " + (cupsPerDay * daysOpen));
        System.out.println("Comfortably inside the range.");
    }
}

RangeCheck.main(null);

---

## Part 5 — Overflow: the negative cup counter

So what happens when a value goes past `Integer.MAX_VALUE`?

It **wraps around** to the most negative value and keeps counting up. Like a car odometer rolling from 999999 back to 000000 — except this odometer rolls into negative numbers.

```
  ... 2147483645, 2147483646, 2147483647, -2147483648, -2147483647, ...
                               ^^^^^^^^^^^  ^^^^^^^^^^^
                                the ceiling   straight to the floor
```

**Critically: there is no error.** No exception, no warning, no crash. The program runs to completion and hands you a wrong number.

**Heads up: the cell below does *not* fail. That's what makes it dangerous.**

In [ ]:
public class Overflow {
    public static void main(String[] args) {
        int max = Integer.MAX_VALUE;
        System.out.println("Largest int:  " + max);
        System.out.println("max + 1:      " + (max + 1));
        System.out.println("max + 2:      " + (max + 2));

        System.out.println();

        int min = Integer.MIN_VALUE;
        System.out.println("Smallest int: " + min);
        System.out.println("min - 1:      " + (min - 1));

        System.out.println();
        System.out.println("Notice: no error message anywhere.");
    }
}

Overflow.main(null);

### The manager's actual bug

Overflow rarely comes from adding 1 to the maximum. It comes from **multiplication**, where numbers grow far faster than people expect.

In [ ]:
public class CupCounter {
    public static void main(String[] args) {
        int cupsPerHour = 250;
        int hours       = 10000000;      // a very optimistic business plan

        int total = cupsPerHour * hours;             // both int -> int result

        System.out.println("Expected:          2500000000");
        System.out.println("What Java printed: " + total);

        // Keeping the calculation in double territory avoids the ceiling
        double safeTotal = (double) cupsPerHour * hours;
        System.out.println("Using a double:    " + safeTotal);
    }
}

CupCounter.main(null);

There's the negative counter. `250 * 10000000` is 2.5 billion, which sails past the 2.147 billion ceiling and wraps into negative territory.

Look at the fix carefully: `(double) cupsPerHour * hours` casts **before** the multiplication, so the math happens in `double` space where the ceiling is astronomically higher. Casting afterward — `(double)(cupsPerHour * hours)` — would be too late, exactly like the division trap in Part 3.

### Which error type is this?

Compiles fine. Runs fine. Wrong answer. That's a **logic error** — the same category as integer division and the empty `Scanner` line. Three lessons in a row, the most dangerous bugs have been the ones that don't announce themselves.

**How to avoid overflow:**
- Estimate the magnitude before you code. Will this ever exceed ~2.1 billion?
- Use `double` for quantities that could get very large.
- Watch multiplication and repeated accumulation especially closely.

---

## Part 6 — Precision: when `double` isn't exact either

`double` has a much bigger range than `int`, so overflow is rarely the problem. Its limitation is different: **`double` stores an approximation.**

A `double` holds roughly 15–17 significant digits in binary. Some decimal values simply cannot be written exactly in binary, the same way `1/3` cannot be written exactly in decimal — `0.333...` never terminates no matter how much paper you have.

`0.1` is one of those values. You saw the symptom back in 1.2. Now here's the consequence.

In [ ]:
public class Precision {
    public static void main(String[] args) {
        System.out.println("0.1 + 0.2 = " + (0.1 + 0.2));
        System.out.println("Does that equal 0.3? " + (0.1 + 0.2 == 0.3));

        System.out.println();

        double total = 0.0;
        total = total + 0.1;
        total = total + 0.1;
        total = total + 0.1;
        System.out.println("0.1 added three times = " + total);
        System.out.println("Does that equal 0.3? " + (total == 0.3));
    }
}

Precision.main(null);

Two lessons in that output.

**First: tiny errors accumulate.** Each `+ 0.1` adds a microscopic amount of inaccuracy. Run that in a loop ten thousand times and the drift becomes visible in dollars.

**Second, and this is the exam-relevant one: never compare `double` values with `==`.** Two calculations that are mathematically identical can differ in the last bit and report `false`. The standard approach is to check whether the difference is *small enough*:

```java
// instead of:  if (a == b)
// professionals write:  if (Math.abs(a - b) < 0.0001)
```

(`Math.abs` is Topic 1.11, and `if` is Unit 2 — this is a preview, not something you need today.)

### The three accuracy limits to know for 1.5.C

| Limit | What causes it | Example |
|---|---|---|
| **Integer division truncation** | `int / int` discards the remainder | `25 / 2` gives `12` |
| **Casting truncation** | `(int)` chops the fractional part | `(int) 5.99` gives `5` |
| **Roundoff error** | Some decimals can't be stored exactly in binary | `0.1 + 0.2` isn't exactly `0.3` |

And a fourth from Part 5 — **overflow** — which is a range problem rather than an accuracy problem, but produces wrong answers just as quietly.

> Remember the note from 1.2: real payment systems dodge roundoff entirely by storing money as a whole number of **cents** in an `int`, never as a `double` of dollars.

---

## Part 7 — The cast decision guide

```
Do I need a decimal result from int values?
      --> cast ONE operand BEFORE the operation:  (double) a / b

Do I need a whole number from a decimal?
      Chopping is fine (how many full cups fit)?   --> (int) value
      I need the nearest whole number?             --> (int)(value + 0.5)

Could this calculation exceed ~2.1 billion?
      --> cast to double BEFORE the arithmetic:    (double) a * b

Am I comparing two doubles for equality?
      --> don't use ==; compare the difference to a small tolerance
```

The pattern underneath all four: **cast early, not late.** Once precision or range is lost, no cast can bring it back.

---

# Practice: Auditing the Register

Four tasks, in order.

---

## Hack 1 — Predict, then run

Fill in every prediction **before** running the cell. Double-click to edit.

| # | Expression | Your prediction | Actual |
|---|---|---|---|
| 1 | `(int) 9.99` | | |
| 2 | `(int) -9.99` | | |
| 3 | `(int)(9.99 + 0.5)` | | |
| 4 | `(double) 9 / 2` | | |
| 5 | `(double) (9 / 2)` | | |
| 6 | `(int) 4.5 + (int) 4.5` | | |

In [ ]:
public class PredictCasts {
    public static void main(String[] args) {
        System.out.println("1. " + ((int) 9.99));
        System.out.println("2. " + ((int) -9.99));
        System.out.println("3. " + ((int)(9.99 + 0.5)));
        System.out.println("4. " + ((double) 9 / 2));
        System.out.println("5. " + ((double) (9 / 2)));
        System.out.println("6. " + ((int) 4.5 + (int) 4.5));
    }
}

PredictCasts.main(null);

<details>
<summary><b>Explanations</b></summary>

1. `9` — truncation chops `.99`, no rounding.
2. `-9` — truncation moves **toward zero**, so it's `-9`, not `-10`.
3. `10` — adding `0.5` gives `10.49`, which truncates to `10`. This is the rounding trick.
4. `4.5` — the cast applies to the `9` first, so it's `9.0 / 2`, a double division.
5. `4.0` — the parentheses force `9 / 2` to run as int division giving `4`, and casting afterward just adds `.0`. **The precision was already gone.**
6. `8` — each `4.5` truncates to `4` *before* the addition, so it's `4 + 4`. Not `9`.

Row 6 is the sneaky one. If you predicted `9`, you added first and cast second — but the casts bind tighter than `+`.
</details>

---

## Hack 2 — Fix the loyalty report

The cafe gives 1 point per cent spent. A customer spent **$17.50** across **4 drinks**.

Correct output should be:

```
Spent:      $17.5
Avg cents:  437.5
Avg points: 438
```

The cell below has **three** bugs. All three compile and run — every number printed is wrong. Fix them, then fill in the report.

In [ ]:
public class BrokenLoyalty {
    public static void main(String[] args) {
        int totalSpentCents = 1750;      // $17.50
        int drinksBought    = 4;

        double dollarsSpent = totalSpentCents / 100;

        double avgCents = (double) (totalSpentCents / drinksBought);

        int avgPoints = (int) avgCents;

        System.out.println("Spent:      $" + dollarsSpent);
        System.out.println("Avg cents:  " + avgCents);
        System.out.println("Avg points: " + avgPoints);
    }
}

BrokenLoyalty.main(null);

**Your bug report** (double-click to edit):

| Bug | What went wrong | Your fix |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |

<details>
<summary><b>Check your answers</b></summary>

**Bug 1 — integer division.** `1750 / 100` is `int / int`, giving `17`, which then widens to `17.0`. Fix by making one operand a double:

```java
double dollarsSpent = totalSpentCents / 100.0;
```

**Bug 2 — the cast is in the wrong place.** `(double) (1750 / 4)` divides first as ints, giving `437`, then casts to `437.0`. The `.5` was destroyed before the cast. Fix by casting an operand instead of the result:

```java
double avgCents = (double) totalSpentCents / drinksBought;
```

**Bug 3 — truncating when rounding was wanted.** `(int) 437.5` chops to `437`. Fix with the rounding trick:

```java
int avgPoints = (int)(avgCents + 0.5);
```

All three are the same underlying mistake: **losing precision before you meant to.** Bugs 1 and 2 lose it in the division; bug 3 loses it in the cast.
</details>

---

## Hack 3 — Investigate the overflow

Answer these in the markdown below, using the cell to experiment.

1. `Integer.MAX_VALUE` is 2,147,483,647. The cafe sells 250 cups per hour. How many hours would it take before an `int` counter overflows? (Work it out, then check with code.)
2. Run the cell as written. Why is the result negative?
3. Modify the cell so it produces the correct value.
4. Which error type is overflow — compile-time, run-time, or logic? How do you know?

In [ ]:
public class OverflowLab {
    public static void main(String[] args) {
        int cupsPerHour = 250;
        int hours       = 10000000;

        int total = cupsPerHour * hours;
        System.out.println("Total cups: " + total);

        // TODO: print how many hours it takes to reach Integer.MAX_VALUE
        //       (hint: divide MAX_VALUE by cupsPerHour)

        // TODO: recompute total so the answer is correct
    }
}

OverflowLab.main(null);

**Your answers** (double-click to edit):

1.

2.

3.

4.

<details>
<summary><b>Check your answers</b></summary>

1. `2147483647 / 250` is about **8,589,934 hours** — roughly 980 years of nonstop service. Comfortable for a cafe, but the point is that the ceiling exists and multiplication finds it fast.

2. `250 * 10000000` is 2,500,000,000, which exceeds the ceiling by about 353 million. The value wraps past `Integer.MAX_VALUE` around to the negative end, giving `-1794967296`.

3. Cast **before** the multiplication so the math happens in `double` space:

```java
double total = (double) cupsPerHour * hours;
```

Writing `(double)(cupsPerHour * hours)` would not work — the overflow happens inside the parentheses, and casting the already-broken result just adds `.0`.

4. **Logic error.** It compiled, it ran to completion, and it printed a number without any error message. Only a human who knows cup counts can't be negative would catch it.
</details>

---

## Hack 4 — Split the bill into dollars and cents

Write a program that takes a bill amount as a `double` and prints the whole dollars and the leftover cents as two separate `int` values.

For `23.87` it should print:

```
Whole dollars: 23
Leftover cents: 87
```

Requirements:

- Get the whole dollars using a cast
- Get the cents from what remains, as an `int`
- Use the rounding trick on the cents

That last requirement is not optional busywork. Once it works, change `bill` to **`4.35`** and run it both with and without the `+ 0.5`. The two answers differ by a penny, and the reason is Part 6.

In [ ]:
public class SplitBill {
    public static void main(String[] args) {
        double bill = 23.87;

        // TODO 1: whole dollars, using a cast

        // TODO 2: the leftover fraction of a dollar

        // TODO 3: convert that fraction to cents as an int, using the rounding trick

        // TODO 4: print both values

        // TODO 5: try TODO 3 again WITHOUT the + 0.5 and note what changes
    }
}

SplitBill.main(null);

<details>
<summary><b>One possible solution</b></summary>

```java
public class SplitBill {
    public static void main(String[] args) {
        double bill = 23.87;

        int wholeDollars = (int) bill;                  // 23
        double fraction  = bill - wholeDollars;         // about 0.8700000000000010
        int cents        = (int)(fraction * 100 + 0.5); // 87

        System.out.println("Whole dollars: " + wholeDollars);
        System.out.println("Leftover cents: " + cents);

        System.out.println("Without the +0.5: " + (int)(fraction * 100));
    }
}

SplitBill.main(null);
```

**Why the `+ 0.5` is essential here.** `23.87` isn't stored exactly — the nearest `double` is a hair off. Subtracting `23` leaves `0.870000000000001`, and multiplying by 100 gives `87.0000000000001`. Here the drift lands *above* the whole number, so truncation happens to give `87` either way.

Now change the bill to **`4.35`** and run both versions:

```java
double bill = 4.35;
// without the +0.5 -> 34 cents        <-- WRONG, a penny short
// with    the +0.5 -> 35 cents        <-- correct
```

This time the drift lands *just below* `35`, at about `34.99999999999999`, and truncation chops it to `34`. Same code, same logic, and the cafe undercharges by a penny on every transaction like this one.

The rounding trick absorbs that drift. This is roundoff error (Part 6) and truncation (Part 2) combining into a real bug that costs real money — and the reason serious payment systems store cents as `int` from the start.
</details>

---

# Self-Check: AP-style questions

**1.** What is the value of `(int) 7.89`?

&nbsp;&nbsp;(A) `7` &nbsp;&nbsp; (B) `8` &nbsp;&nbsp; (C) `7.0` &nbsp;&nbsp; (D) A compile-time error

<details><summary>Answer</summary>

**(A) 7**. Casting to `int` truncates — it discards the fractional part rather than rounding. The result is an `int`, so it has no `.0`.
</details>

---

**2.** Given `int a = 9;` and `int b = 4;`, which expression evaluates to `2.25`?

&nbsp;&nbsp;(A) `a / b`
&nbsp;&nbsp;(B) `(double) (a / b)`
&nbsp;&nbsp;(C) `(double) a / b`
&nbsp;&nbsp;(D) `(int) a / b`

<details><summary>Answer</summary>

**(C)**. The cast applies to `a` before the division, so it becomes `9.0 / 4`. (A) is int division giving `2`. (B) divides first as ints and casts the `2` afterward, giving `2.0` — the precision was already lost. (D) casts an `int` to `int`, changing nothing.
</details>

---

**3.** What is printed?

```java
int x = Integer.MAX_VALUE;
System.out.println(x + 1);
```

&nbsp;&nbsp;(A) `2147483648`
&nbsp;&nbsp;(B) `-2147483648`
&nbsp;&nbsp;(C) `0`
&nbsp;&nbsp;(D) The program throws an exception

<details><summary>Answer</summary>

**(B) `-2147483648`**. The value exceeds the `int` ceiling and wraps around to `Integer.MIN_VALUE`. Note especially that **no exception is thrown** — overflow is silent, which is what makes it dangerous.
</details>

---

**4.** What is the value of `(int) -3.7`?

&nbsp;&nbsp;(A) `-4` &nbsp;&nbsp; (B) `-3` &nbsp;&nbsp; (C) `3` &nbsp;&nbsp; (D) `-3.0`

<details><summary>Answer</summary>

**(B) `-3`**. Truncation removes the fractional digits, which moves the value **toward zero**. It is not the same as rounding down — rounding down would give `-4`.
</details>

---

**5.** Which expression correctly rounds the `double` variable `avg` to the nearest whole number, assuming `avg` is positive?

&nbsp;&nbsp;(A) `(int) avg`
&nbsp;&nbsp;(B) `(int) avg + 0.5`
&nbsp;&nbsp;(C) `(int)(avg + 0.5)`
&nbsp;&nbsp;(D) `(double)(avg + 0.5)`

<details><summary>Answer</summary>

**(C)**. The addition must happen **before** the cast, so it needs its own parentheses. (B) casts first and then adds `0.5`, producing a `double` — and it truncated before rounding could help. (A) truncates. (D) never becomes a whole number.
</details>

---

**6.** Why does `0.1 + 0.2 == 0.3` evaluate to `false` in Java?

&nbsp;&nbsp;(A) `==` cannot be used on numbers
&nbsp;&nbsp;(B) Some decimal values cannot be represented exactly as a `double`, so the sum is slightly off
&nbsp;&nbsp;(C) Java rounds all doubles to one decimal place
&nbsp;&nbsp;(D) The expression causes an overflow

<details><summary>Answer</summary>

**(B)**. A `double` stores a binary approximation, and `0.1` has no exact binary representation. The sum comes out as about `0.30000000000000004`. This is why `double` values should be compared with a small tolerance rather than `==`.
</details>

---

**7.** A program computes `int seats = cupsPerHour * hours;` and prints a negative number, though both variables are positive. There is no error message. What happened, and what type of error is it?

&nbsp;&nbsp;(A) A run-time error caused by division by zero
&nbsp;&nbsp;(B) A compile-time error the programmer ignored
&nbsp;&nbsp;(C) Integer overflow, which is a logic error
&nbsp;&nbsp;(D) A roundoff error in the `double` representation

<details><summary>Answer</summary>

**(C)**. The product exceeded `Integer.MAX_VALUE` and wrapped into negative values. Because the program compiled, ran to completion, and produced no error message while giving a wrong answer, it is a **logic error**. (D) is wrong because roundoff affects `double`, not `int`.
</details>

---

**8.** Which of the following does **not** cause a loss of accuracy?

&nbsp;&nbsp;(A) `int result = (int) 9.99;`
&nbsp;&nbsp;(B) `int result = 25 / 4;`
&nbsp;&nbsp;(C) `double result = 7;`
&nbsp;&nbsp;(D) `double result = 0.1 + 0.2;`

<details><summary>Answer</summary>

**(C)**. Widening an `int` into a `double` is always exact — `7` becomes `7.0` with nothing lost. (A) truncates, (B) is integer division, and (D) introduces roundoff error.
</details>

---

# Closing time

### Vocabulary to know cold

| Term | One-line definition |
|---|---|
| Cast | `(type) value` — explicitly converting a value to another primitive type |
| Widening | `int` → `double`; automatic and lossless |
| Narrowing | `double` → `int`; requires a cast and discards the fraction |
| Truncation | Removing the fractional digits, moving the value toward zero |
| `Integer.MAX_VALUE` | 2,147,483,647 — the largest `int` |
| `Integer.MIN_VALUE` | −2,147,483,648 — the smallest `int` |
| Overflow | Exceeding a type's range; the value wraps around with no error |
| Roundoff error | Inaccuracy from decimals that can't be stored exactly in binary |

### The six things that will show up on the exam

1. `(int)` **truncates**, never rounds. `(int) 5.99` is `5`.
2. Truncation moves toward **zero**. `(int) -5.99` is `-5`.
3. A cast binds tighter than `* / % + -`. `(double) a / b` and `(double)(a / b)` are different.
4. Cast **before** the operation. Precision lost is precision gone.
5. Overflow wraps silently past ±2.1 billion — **no exception**.
6. Never compare `double` values with `==`.

### Before you submit, check that you:

- [ ] Ran every code cell top to bottom
- [ ] Filled in all six predictions in Hack 1 **before** running it
- [ ] Fixed all three bugs in Hack 2 and got `$17.5`, `437.5`, `438`
- [ ] Answered all four questions in Hack 3
- [ ] Completed Hack 4, including running it once **without** the `+ 0.5`
- [ ] Attempted all eight self-check questions before revealing answers

### Next up

**1.6 — Compound Assignment Operators.** You've written `total = total + price` a dozen times across the last two lessons. Java has a shorthand for it — `total += price` — along with `-=`, `*=`, `/=`, `%=`, and the increment and decrement operators `++` and `--`. Short, extremely common on the exam, and it hides one genuinely nasty surprise involving `/=` on integers.

See you at the next shift 🧋